# Sparse-MeZO on LeNet-5 / MNIST: results from the stored weights

This notebook scores the stored weight files on the noisy MNIST test set and writes the results to an Excel workbook.
It trains nothing and reads no other notebook, script or results file: everything it needs is in the cells below.

| Row | Weight file, for input-noise variance *v* |
|---|---|
| Full retrain | `weights_baselines/full_retrain_var_<v>.pth` |
| Init (no retrain) | `weights_baselines/full_retrain_var_0.0.pth`, the clean model every run starts from |
| BP-FC | `weights_baselines/bp_fc_var_<v>.pth` |
| Sparse-MeZO(10ep) | `weights_smezo_spsa_lenet5_mnist/smezo_spsa_var_<v>.pth` |

- **Test set:** MNIST train and test (70,000 images) pooled and split 60/10/30 per class with seed 42, which
  gives 20,999 test images. Gaussian noise of variance *v* is added after normalisation, drawn on the CPU with seed 44
  (42 + 2, the test split's noise stream).
- **Metrics:** accuracy, macro precision, recall and F1, and mAP (the mean one-vs-rest average precision on the softmax
  scores), rounded to 6 decimals. Accuracies can differ by a few test images between machines (GPU and CPU arithmetic).
- **Output:** `lenet5_mnist/results_smezo_spsa_lenet5_mnist/results_smezo_spsa_lenet5_mnist_reproduced.xlsx`, one sheet `long`
  with a row per noise level and method; a re-run overwrites it. The name carries no `YYYYmmdd_HHMMSS` stamp, so the
  plotting notebooks, which read the workbook with the newest stamp in each results folder, keep using the reported one.

**How to run:** run all cells. The notebook looks for the project folder (the one holding `lenet5_mnist/` and `data/`)
from its own folder upwards and reads MNIST from `data/MNIST`, downloading it there if it is missing. It needs torch,
torchvision, numpy, pandas and openpyxl.

In [1]:
import math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms

## Settings

In [2]:
# ---- settings: edit here if needed ----
DATASET = "lenet5_mnist"
METHOD_KEY = "smezo_spsa"              # the weights folder is weights_<METHOD_KEY>_lenet5_mnist/
METHOD_LABEL = "Sparse-MeZO(10ep)"     # the method's row label in the workbook

DEVICE = "cpu"                         # "cuda" works too, on a GPU this torch build supports
BATCH_SIZE = 1024                      # evaluation batch; lower it if memory is short
SEED = 42
NOISE_LEVELS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]   # input-noise variances
SPLIT_FRACS = (0.60, 0.10, 0.30)       # train / validation / test fractions of the pooled data, per class
TEST_NOISE_SEED = SEED + 2             # the test split's noise stream
NUM_CLASSES = 10


def find_project_root(start=Path.cwd()):
    """The BOLSTER folder: the first folder, from the notebook's folder upwards, that holds lenet5_mnist/ and data/."""
    for c in (start, *start.parents):
        if (c / DATASET).is_dir() and (c / "data").is_dir():
            return c
    raise FileNotFoundError(f"no folder with {DATASET}/ and data/ above {start}")


PROJECT = find_project_root()
DATA_ROOT = PROJECT / "data"
BASELINE_WEIGHTS = PROJECT / DATASET / "weights_baselines"
METHOD_WEIGHTS = PROJECT / DATASET / f"weights_{METHOD_KEY}_{DATASET}"
RESULTS_DIR = PROJECT / DATASET / f"results_{METHOD_KEY}_{DATASET}"
# no YYYYmmdd_HHMMSS stamp in the name: the plotting notebooks keep reading the reported workbook
XLSX_PATH = RESULTS_DIR / f"results_{METHOD_KEY}_{DATASET}_reproduced.xlsx"

print(f"project: {PROJECT}\ndevice: {DEVICE}")

project: /home/rahman2/GIT/BOLSTER
device: cpu


## The network

In [3]:
class LeNet5(nn.Module):
    """LeNet-5 for 28x28x1 MNIST with ReLU and max-pool: conv1(6) - pool - conv2(16) - pool - conv3/C5(120) - f6(84) -
    fc(10). The layer names match the stored state dicts."""
    def __init__(self, num_classes=NUM_CLASSES, in_ch=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, 6, 5, padding=2)        # 28x28 -> 28x28
        self.conv2 = nn.Conv2d(6, 16, 5)                      # 14x14 -> 10x10
        self.conv3 = nn.Conv2d(16, 120, 5)                    # 5x5   -> 1x1  (C5)
        self.f6    = nn.Linear(120, 84)
        self.fc    = nn.Linear(84, num_classes)

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)            # -> 14x14
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)            # -> 5x5
        x = F.relu(self.conv3(x))                             # -> 120x1x1
        x = x.flatten(1)                                      # -> 120
        x = F.relu(self.f6(x))
        return self.fc(x)

## Metrics

In [4]:
def macro_prf(y_true, y_pred, num_classes=NUM_CLASSES):
    """Macro precision, recall and F1 over the classes; a class with an empty denominator contributes 0."""
    cm = np.zeros((num_classes, num_classes), dtype=np.int64)
    np.add.at(cm, (y_true, y_pred), 1)
    tp = np.diag(cm).astype(np.float64)
    fp = cm.sum(0) - tp
    fn = cm.sum(1) - tp
    prec = np.divide(tp, tp + fp, out=np.zeros_like(tp), where=(tp + fp) > 0)
    rec  = np.divide(tp, tp + fn, out=np.zeros_like(tp), where=(tp + fn) > 0)
    den  = prec + rec
    f1   = np.divide(2 * prec * rec, den, out=np.zeros_like(tp), where=den > 0)
    return float(prec.mean()), float(rec.mean()), float(f1.mean())


def average_precision(is_pos, scores):
    """One-vs-rest average precision of one class (the scikit-learn definition: the sum over the distinct score
    thresholds of (R_n - R_{n-1}) P_n)."""
    n_pos = int(is_pos.sum())
    if n_pos == 0:
        return 0.0
    order = np.argsort(-scores, kind="mergesort")
    y, s = is_pos[order], scores[order]
    last = np.r_[np.where(np.diff(s))[0], y.size - 1]        # the last index of each distinct score
    tps = np.cumsum(y)[last]
    fps = (1 + last) - tps
    precision = np.r_[1.0, tps / (tps + fps)]
    recall    = np.r_[0.0, tps / n_pos]
    return float(np.sum(np.diff(recall) * precision[1:]))


@torch.no_grad()
def eval_metrics(state_dict, x, y, model_cls=LeNet5):
    """accuracy, macro precision / recall / F1 and mAP of the model in state_dict on (x, y). The argmax gives the
    predictions, the softmax the scores for mAP."""
    model = model_cls().to(DEVICE)
    model.load_state_dict(state_dict)
    model.eval()
    probs, preds = [], []
    for i in range(0, y.numel(), BATCH_SIZE):
        logits = model(x[i:i + BATCH_SIZE].to(DEVICE))
        probs.append(F.softmax(logits, dim=1).cpu())
        preds.append(logits.argmax(1).cpu())
    probs, y_pred, y_true = torch.cat(probs).numpy(), torch.cat(preds).numpy(), y.numpy()
    precision, recall, f1 = macro_prf(y_true, y_pred)
    mAP = float(np.mean([average_precision((y_true == c).astype(np.int64), probs[:, c]) for c in range(NUM_CLASSES)]))
    return {"accuracy": float((y_pred == y_true).mean()), "precision": precision, "recall": recall, "f1": f1,
            "map": mAP}

## The test set
The pooled 60/10/30 split; only its test part is used.

In [5]:
tfm = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])


def stack(ds):
    xs, ys = zip(*((x, y) for x, y in ds))
    return torch.stack(xs), torch.tensor(ys, dtype=torch.long)


def add_noise(x, var, seed):
    """x plus Gaussian noise of variance var, drawn on the CPU from a generator seeded with seed."""
    if var <= 0:
        return x
    g = torch.Generator().manual_seed(seed)
    return x + torch.randn(x.shape, generator=g) * float(np.sqrt(var))


def stratified_split(y, fracs=SPLIT_FRACS, seed=SEED):
    """Per class: a seeded shuffle, then the first round(f_train n_c) samples go to train, the next round(f_val n_c) to
    validation, the rest to test. Returns the three index tensors into the pool (each sorted)."""
    g = torch.Generator().manual_seed(int(seed))
    parts = ([], [], [])
    for c in torch.unique(y).tolist():
        idx = torch.nonzero(y == c).flatten()
        idx = idx[torch.randperm(idx.numel(), generator=g)]
        n_c = idx.numel()
        n_a, n_b = int(math.floor(fracs[0] * n_c + 0.5)), int(math.floor(fracs[1] * n_c + 0.5))
        parts[0].append(idx[:n_a]); parts[1].append(idx[n_a:n_a + n_b]); parts[2].append(idx[n_a + n_b:])
    return tuple(torch.sort(torch.cat(p)).values for p in parts)


x_tr, y_tr = stack(datasets.MNIST(str(DATA_ROOT), train=True, download=True, transform=tfm))
x_te, y_te = stack(datasets.MNIST(str(DATA_ROOT), train=False, download=True, transform=tfm))
x_pool, y_pool = torch.cat([x_tr, x_te]), torch.cat([y_tr, y_te])   # MNIST train first, then MNIST test
idx_tr, idx_va, idx_te = stratified_split(y_pool)
x_test, y_test = x_pool[idx_te], y_pool[idx_te]
del x_tr, y_tr, x_te, y_te, x_pool
print(f"pool {y_pool.numel():,} = train {idx_tr.numel():,} + validation {idx_va.numel():,} + test {idx_te.numel():,}")

pool 70,000 = train 42,001 + validation 7,000 + test 20,999


## Scores and workbook
Which weight file feeds which row; every row is scored on that level's noisy test set, then the workbook is written.

In [6]:
ROWS = [                               # (row label, weight file for noise variance v), in the workbook's order
    ("Full retrain", lambda v: BASELINE_WEIGHTS / f"full_retrain_var_{v}.pth"),
    ("Init (no retrain)", lambda v: BASELINE_WEIGHTS / "full_retrain_var_0.0.pth"),
    ("BP-FC", lambda v: BASELINE_WEIGHTS / f"bp_fc_var_{v}.pth"),
    (METHOD_LABEL, lambda v: METHOD_WEIGHTS / f"{METHOD_KEY}_var_{v}.pth"),
]
METRIC_COLS = ["accuracy", "precision", "recall", "f1", "map"]


def load_state(path):
    return torch.load(path, map_location="cpu", weights_only=True)


records = []
for v in NOISE_LEVELS:
    level = f"{v:.0%}"
    x_noisy = add_noise(x_test, v, seed=TEST_NOISE_SEED)
    for label, path_of in ROWS:
        path = path_of(v)
        r = eval_metrics(load_state(path), x_noisy, y_test)
        records.append({"noise_level": level, "method": label, **{m: round(r[m], 6) for m in METRIC_COLS}})
        print(f"{level:>5}  {label:<20} acc {r['accuracy']:.6f}  f1 {r['f1']:.6f}  map {r['map']:.6f}   <- {path.name}")

results = pd.DataFrame(records, columns=["noise_level", "method", *METRIC_COLS])
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
results.to_excel(XLSX_PATH, sheet_name="long", index=False)
print(f"\nwrote {XLSX_PATH.relative_to(PROJECT)}")

  10%  Full retrain         acc 0.976094  f1 0.975911  map 0.996403   <- full_retrain_var_0.1.pth


  10%  Init (no retrain)    acc 0.967713  f1 0.967520  map 0.993753   <- full_retrain_var_0.0.pth


  10%  BP-FC                acc 0.968808  f1 0.968599  map 0.994215   <- bp_fc_var_0.1.pth


  10%  Sparse-MeZO(10ep)    acc 0.967856  f1 0.967659  map 0.993763   <- smezo_spsa_var_0.1.pth


  20%  Full retrain         acc 0.974808  f1 0.974641  map 0.996045   <- full_retrain_var_0.2.pth


  20%  Init (no retrain)    acc 0.964046  f1 0.963840  map 0.992436   <- full_retrain_var_0.0.pth


... (30 outputs left out)

 100%  Full retrain         acc 0.952903  f1 0.952608  map 0.988326   <- full_retrain_var_1.0.pth


 100%  Init (no retrain)    acc 0.866565  f1 0.868023  map 0.960554   <- full_retrain_var_0.0.pth


 100%  BP-FC                acc 0.927520  f1 0.926935  map 0.976573   <- bp_fc_var_1.0.pth


 100%  Sparse-MeZO(10ep)    acc 0.886328  f1 0.886981  map 0.963788   <- smezo_spsa_var_1.0.pth

wrote lenet5_mnist/results_smezo_spsa_lenet5_mnist/results_smezo_spsa_lenet5_mnist_reproduced.xlsx
